Dados da loterica da Mega Sena

In [1]:
import pandas as pd

dados retirados daqui https://loterias.caixa.gov.br/Paginas/Mega-Sena.aspx

In [2]:
nome_do_arquivo = "Mega-Sena.xlsx"
df_volume = pd.read_excel(nome_do_arquivo, sheet_name='MEGA SENA')
print("Dados da aba Volume:")
print(df_volume.head())

Dados da aba Volume:
   Concurso Data do Sorteio  Bola1  Bola2  Bola3  Bola4  Bola5  Bola6  \
0         1      11/03/1996      4      5     30     33     41     52   
1         2      18/03/1996      9     37     39     41     43     49   
2         3      25/03/1996     10     11     29     30     36     47   
3         4      01/04/1996      1      5      6     27     42     59   
4         5      08/04/1996      1      2      6     16     19     46   

   Ganhadores 6 acertos Cidade / UF Rateio 6 acertos  Ganhadores 5 acertos  \
0                     0         NaN           R$0,00                    17   
1                     1          PR   R$2.307.162,23                    65   
2                     2      RN; SP     R$391.192,51                    62   
3                     0         NaN           R$0,00                    39   
4                     0         NaN           R$0,00                    98   

  Rateio 5 acertos  Ganhadores 4 acertos Rateio 4 acertos Acumulado 6 a

In [3]:
df_pesquisa =  df_volume[['Data do Sorteio', 'Bola1', 'Bola2', 'Bola3', 'Bola4',
       'Bola5', 'Bola6']]

In [4]:
df_pesquisa.info()

print(df_pesquisa.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2950 entries, 0 to 2949
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data do Sorteio  2950 non-null   object
 1   Bola1            2950 non-null   int64 
 2   Bola2            2950 non-null   int64 
 3   Bola3            2950 non-null   int64 
 4   Bola4            2950 non-null   int64 
 5   Bola5            2950 non-null   int64 
 6   Bola6            2950 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 161.5+ KB
Data do Sorteio    0
Bola1              0
Bola2              0
Bola3              0
Bola4              0
Bola5              0
Bola6              0
dtype: int64


In [5]:
df_pesquisa['Data do Sorteio'] = pd.to_datetime(
    df_pesquisa['Data do Sorteio'],
    format='%d/%m/%Y'  # Informa ao Pandas que o formato é Dia/Mês/Ano
)

/tmp/ipykernel_5363/1612541821.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Data do Sorteio'] = pd.to_datetime(


In [6]:
df_pesquisa.tail()

,Data do Sorteio,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
2945,2025-12-02,4,13,17,21,49,54
2946,2025-12-04,4,10,15,37,39,44
2947,2025-12-06,6,24,37,52,53,58
2948,2025-12-09,4,6,11,38,49,54
2949,2025-12-11,21,23,42,49,50,60



previsão da mega sena pelo prophet, mas necessito de treino e teste para achar o melhor hiperparametro no modelo prophet, com a validação cruzada e, no final, achando o melhor parametro fazer a previsão com todos os dados.

1️⃣ Imports

In [8]:
# import bibliotecas
import itertools
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics


## Estratégia que vamos usar

Para cada bola separadamente:

Usar cross-validation nativa do Prophet

Testar grade de hiperparâmetros

Escolher o melhor pelo RMSE

Re-treinar o Prophet com TODOS os dados

Fazer previsão de 35 dias

Juntar tudo em 1 tabela final

2️⃣ Garantir formato correto

In [9]:
df_pesquisa['Data do Sorteio'] = pd.to_datetime(df_pesquisa['Data do Sorteio'])
df_pesquisa = df_pesquisa.sort_values('Data do Sorteio')


/tmp/ipykernel_5363/392360852.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Data do Sorteio'] = pd.to_datetime(df_pesquisa['Data do Sorteio'])


3️⃣ Grade de hiperparâmetros (enxuta e eficiente)

Prophet não aguenta grids gigantes.

Essa grade já é suficiente para aprendizado sério.

In [10]:
param_grid = {
    'changepoint_prior_scale': [0.01, 0.1, 0.5],
    'seasonality_prior_scale': [1.0, 5.0, 10.0],
    'weekly_seasonality': [True, False],
    'yearly_seasonality': [True, False]
}

all_params = [
    dict(zip(param_grid.keys(), v))
    for v in itertools.product(*param_grid.values())
]


In [11]:
all_params

[{'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 1.0,
  'weekly_seasonality': True,
  'yearly_seasonality': True},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 1.0,
  'weekly_seasonality': True,
  'yearly_seasonality': False},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 1.0,
  'weekly_seasonality': False,
  'yearly_seasonality': True},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 1.0,
  'weekly_seasonality': False,
  'yearly_seasonality': False},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 5.0,
  'weekly_seasonality': True,
  'yearly_seasonality': True},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 5.0,
  'weekly_seasonality': True,
  'yearly_seasonality': False},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 5.0,
  'weekly_seasonality': False,
  'yearly_seasonality': True},
 {'changepoint_prior_scale': 0.01,
  'seasonality_prior_scale': 5.0,
  'weekly_seaso

4️⃣ Função para achar o melhor modelo (CV Prophet)

In [12]:
def prophet_cv_best_model(df_prophet, params_list):
    rmses = []

    for params in params_list:
        m = Prophet(
            changepoint_prior_scale=params['changepoint_prior_scale'],
            seasonality_prior_scale=params['seasonality_prior_scale'],
            weekly_seasonality=params['weekly_seasonality'],
            yearly_seasonality=params['yearly_seasonality'],
            daily_seasonality=False
        )

        m.fit(df_prophet)

        # Cross-validation temporal
        df_cv = cross_validation(
            m,
            initial='365 days',
            period='90 days',
            horizon='90 days',
            parallel="processes"
        )

        df_p = performance_metrics(df_cv)
        rmses.append(df_p['rmse'].mean())

    best_params = params_list[rmses.index(min(rmses))]
    return best_params, min(rmses)


🔎 Ajuste initial / period / horizon se o histórico for menor.

 5️⃣ Encontrar melhor hiperparâmetro para cada bola

In [13]:
bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']

melhores_parametros = {}

for bola in bolas:
    print(f'🔍 Ajustando {bola}...')

    df_prophet = df_pesquisa[['Data do Sorteio', bola]].rename(
        columns={'Data do Sorteio': 'ds', bola: 'y'}
    )

    best_params, best_rmse = prophet_cv_best_model(df_prophet, all_params)

    melhores_parametros[bola] = best_params
    print(f'✅ Melhor RMSE {bola}: {best_rmse:.2f}')


20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] done processing


🔍 Ajustando Bola1...


Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] done processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] start processing
20:03:01 - cmdstanpy - INFO - Chain [1] done processing
20:03:01 - cmdstanpy - INFO - Chain [1] done processing
20:03:01 - cmdstanpy - INFO - Chain [1] done processing
20:03:01 - cmdstanpy - INFO - Chain [1] start

✅ Melhor RMSE Bola1: 6.98
🔍 Ajustando Bola2...


Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] done processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] start processing
20:05:14 - cmdstanpy - INFO - Chain [1] done processing
20:05:14 - cmdstanpy - INFO - Chain [1] done processing
20:05:14 - cmdstanpy - INFO - Chain [1] done processing
20:05:14 - cmdstanpy - INFO - Chain [1] done processing
20:05:14 - cmdstanpy - INFO - Chain [1] start 

...

✅ Melhor RMSE Bola4: 9.86
🔍 Ajustando Bola5...


20:13:04 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] done processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] start processing
20:13:04 - cmdstanpy - INFO - Chain [1] done processing
20:13:04 - cmdstanpy - INFO - Chain [1] done processing
20:13:04 - cmdstanpy - INFO - Chain [1] done 

✅ Melhor RMSE Bola5: 9.00
🔍 Ajustando Bola6...


Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] done processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] done processing
20:15:37 - cmdstanpy - INFO - Chain [1] done processing
20:15:37 - cmdstanpy - INFO - Chain [1] done processing
20:15:37 - cmdstanpy - INFO - Chain [1] start processing
20:15:37 - cmdstanpy - INFO - Chain [1] done 

✅ Melhor RMSE Bola6: 6.95


 6️⃣ Treinar modelo FINAL com TODOS os dados + previsão 35 dias

In [16]:
for bola in bolas:
    params = melhores_parametros[bola]
    print(bola)
    print(params)

Bola1
{'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}
Bola2
{'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}
Bola3
{'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}
Bola4
{'changepoint_prior_scale': 0.01, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}
Bola5
{'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}
Bola6
{'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 1.0, 'weekly_seasonality': False, 'yearly_seasonality': False}


In [14]:
resultados = []

for bola in bolas:
    params = melhores_parametros[bola]

    df_prophet = df_pesquisa[['Data do Sorteio', bola]].rename(
        columns={'Data do Sorteio': 'ds', bola: 'y'}
    )

    model = Prophet(
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        weekly_seasonality=params['weekly_seasonality'],
        yearly_seasonality=params['yearly_seasonality'],
        daily_seasonality=False
    )

    model.fit(df_prophet)

    future = model.make_future_dataframe(periods=35)
    forecast = model.predict(future)

    previsao_futura = forecast[['ds', 'yhat']].tail(35)
    previsao_futura['Bola'] = bola

    resultados.append(previsao_futura)


20:20:04 - cmdstanpy - INFO - Chain [1] start processing
20:20:04 - cmdstanpy - INFO - Chain [1] done processing
20:20:04 - cmdstanpy - INFO - Chain [1] start processing
20:20:04 - cmdstanpy - INFO - Chain [1] done processing
20:20:05 - cmdstanpy - INFO - Chain [1] start processing
20:20:05 - cmdstanpy - INFO - Chain [1] done processing
20:20:05 - cmdstanpy - INFO - Chain [1] start processing
20:20:05 - cmdstanpy - INFO - Chain [1] done processing
20:20:05 - cmdstanpy - INFO - Chain [1] start processing
20:20:05 - cmdstanpy - INFO - Chain [1] done processing
20:20:05 - cmdstanpy - INFO - Chain [1] start processing
20:20:06 - cmdstanpy - INFO - Chain [1] done processing


 7️⃣ Tabela final: 1 dia → 6 bolas

In [15]:
previsoes = pd.concat(resultados)

previsoes_final = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat'
).reset_index()

for bola in bolas:
    previsoes_final[bola] = (
        previsoes_final[bola]
        .round()
        .clip(1, 60)
        .astype(int)
    )

previsoes_final.head()


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9,18,27,35,44,53
1,2025-12-13,9,18,27,35,44,53
2,2025-12-14,9,18,27,35,44,53
3,2025-12-15,9,18,27,35,44,53
4,2025-12-16,9,18,27,35,44,53


In [17]:
display(previsoes_final)

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9,18,27,35,44,53
1,2025-12-13,9,18,27,35,44,53
2,2025-12-14,9,18,27,35,44,53
3,2025-12-15,9,18,27,35,44,53
4,2025-12-16,9,18,27,35,44,53
5,2025-12-17,9,18,27,35,44,53
6,2025-12-18,9,18,27,35,44,53
7,2025-12-19,9,18,27,35,44,53
8,2025-12-20,9,18,27,35,44,53
9,2025-12-21,9,18,27,35,44,53


In [18]:
resultados = []

for bola in bolas:
    params = melhores_parametros[bola]

    df_prophet = df_pesquisa[['Data do Sorteio', bola]].rename(
        columns={'Data do Sorteio': 'ds', bola: 'y'}
    )

    model = Prophet(
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        weekly_seasonality=True, # correção considerar a sazonalidade
        yearly_seasonality=True,
        daily_seasonality=False
    )

    model.fit(df_prophet)

    future = model.make_future_dataframe(periods=35)
    forecast = model.predict(future)

    previsao_futura = forecast[['ds', 'yhat']].tail(35)
    previsao_futura['Bola'] = bola

    resultados.append(previsao_futura)


20:27:53 - cmdstanpy - INFO - Chain [1] start processing
20:27:53 - cmdstanpy - INFO - Chain [1] done processing
20:27:53 - cmdstanpy - INFO - Chain [1] start processing
20:27:54 - cmdstanpy - INFO - Chain [1] done processing
20:27:54 - cmdstanpy - INFO - Chain [1] start processing
20:27:54 - cmdstanpy - INFO - Chain [1] done processing
20:27:54 - cmdstanpy - INFO - Chain [1] start processing
20:27:54 - cmdstanpy - INFO - Chain [1] done processing
20:27:55 - cmdstanpy - INFO - Chain [1] start processing
20:27:55 - cmdstanpy - INFO - Chain [1] done processing
20:27:55 - cmdstanpy - INFO - Chain [1] start processing
20:27:55 - cmdstanpy - INFO - Chain [1] done processing


In [19]:
previsoes = pd.concat(resultados)

previsoes_final = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat'
).reset_index()

for bola in bolas:
    previsoes_final[bola] = (
        previsoes_final[bola]
        .round()
        .clip(1, 60)
        .astype(int)
    )

previsoes_final.head()


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9,18,24,30,40,51
1,2025-12-13,10,20,26,35,43,52
2,2025-12-14,9,18,24,35,42,51
3,2025-12-15,8,21,27,34,40,51
4,2025-12-16,9,20,26,36,44,52


In [20]:
previsoes_final

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9,18,24,30,40,51
1,2025-12-13,10,20,26,35,43,52
2,2025-12-14,9,18,24,35,42,51
3,2025-12-15,8,21,27,34,40,51
4,2025-12-16,9,20,26,36,44,52
5,2025-12-17,9,19,26,35,43,52
6,2025-12-18,10,19,28,36,43,52
7,2025-12-19,9,18,25,31,41,51
8,2025-12-20,9,20,28,36,44,52
9,2025-12-21,8,18,25,36,44,51


In [24]:
previsao_futura

,ds,yhat,Bola
2950,2025-12-12,51.196383,Bola6
2951,2025-12-13,51.940624,Bola6
2952,2025-12-14,51.131643,Bola6
2953,2025-12-15,51.339950,Bola6
2954,2025-12-16,52.255650,Bola6
2955,2025-12-17,51.569236,Bola6
2956,2025-12-18,52.005086,Bola6
2957,2025-12-19,51.427484,Bola6
2958,2025-12-20,52.227554,Bola6
2959,2025-12-21,51.468807,Bola6


In [25]:
resultados

[             ds      yhat   Bola
 2950 2025-12-12  9.347576  Bola1
 2951 2025-12-13  9.607058  Bola1
 2952 2025-12-14  8.737672  Bola1
 2953 2025-12-15  8.463355  Bola1
 2954 2025-12-16  9.289499  Bola1
 2955 2025-12-17  9.372734  Bola1
 2956 2025-12-18  9.688075  Bola1
 2957 2025-12-19  9.278188  Bola1
 2958 2025-12-20  9.419236  Bola1
 2959 2025-12-21  8.436423  Bola1
 2960 2025-12-22  8.056123  Bola1
 2961 2025-12-23  8.785966  Bola1
 2962 2025-12-24  8.784559  Bola1
 2963 2025-12-25  9.028594  Bola1
 2964 2025-12-26  8.562064  Bola1
 2965 2025-12-27  8.662085  Bola1
 2966 2025-12-28  7.654416  Bola1
 2967 2025-12-29  7.265580  Bola1
 2968 2025-12-30  8.002947  Bola1
 2969 2025-12-31  8.024475  Bola1
 2970 2026-01-01  8.305829  Bola1
 2971 2026-01-02  7.889642  Bola1
 2972 2026-01-03  8.051369  Bola1
 2973 2026-01-04  7.114859  Bola1
 2974 2026-01-05  6.804534  Bola1
 2975 2026-01-06  7.625529  Bola1
 2976 2026-01-07  7.733497  Bola1
 2977 2026-01-08  8.101796  Bola1
 2978 2026-01-